In [1]:
!pip install transformers datasets evaluate seqeval accelerate torch

In [2]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)

In [3]:
df = pd.read_csv("clean_jobs.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10100, 12)


,job_id,job_title,company,location,job_description,experience,education,salary,job_type,clean_description,original_length,clean_length
0,JOB07402,Frontend Developer,DataBridge Analytics,"Gurgaon, India",We are looking for a Frontend Developer to joi...,1-3 years,B.E.,9-14 LPA,Full-time,we are looking for a frontend developer to joi...,362,351
1,JOB05835,Software Engineer,Vertex Digital,"Mumbai, India",We are looking for a Software Engineer to join...,3-5 years,Bachelor's Degree,12-16 LPA,Internship,we are looking for a software engineer to join...,342,333
2,JOB02123,Machine Learning Engineer,CloudSphere,"Mumbai, India",We are looking for a Machine Learning Engineer...,1-3 years,B.E.,8-11 LPA,Full-time,we are looking for a machine learning engineer...,364,354
3,JOB08789,Cloud Engineer,Apex Solutions,"Mumbai, India",We are looking for a Cloud Engineer to join ou...,3-5 years,Master's Degree,11-16 LPA,Full-time,we are looking for a cloud engineer to join ou...,361,350
4,JOB00305,Frontend Developer,Quantix Technologies,"Noida, India",We are looking for a Frontend Developer to joi...,3-5 years,B.Tech,10-12 LPA,Full-time,we are looking for a frontend developer to joi...,366,355


In [4]:
skills = [
    "Python",
    "Java",
    "C++",
    "JavaScript",
    "SQL",
    "MySQL",
    "PostgreSQL",
    "MongoDB",
    "Power BI",
    "Tableau",
    "Excel",
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "TensorFlow",
    "PyTorch",
    "AWS",
    "Azure",
    "GCP",
    "Spark",
    "Hadoop",
    "Docker",
    "Kubernetes",
    "Git",
    "Machine Learning",
    "Deep Learning",
    "Natural Language Processing",
    "Data Science",
    "Computer Vision"
]

skills_lower = [skill.lower() for skill in skills]

In [5]:
import re

def create_bio_example(text):
    words = str(text).split()
    labels = ["O"] * len(words)

    text_lower = str(text).lower()

    for skill in skills:
        skill_words = skill.lower().split()

        for i in range(len(words) - len(skill_words) + 1):

            phrase = " ".join(
                re.sub(r"[^a-zA-Z0-9+#.-]", "", words[i+j]).lower()
                for j in range(len(skill_words))
            )

            target = " ".join(skill_words)

            if phrase == target:
                labels[i] = "B-SKILL"

                for j in range(1, len(skill_words)):
                    labels[i+j] = "I-SKILL"

    return words, labels

In [6]:
test_text = "Experience with Python and Machine Learning"

words, labels = create_bio_example(test_text)

for word, label in zip(words, labels):
    print(word, "->", label)

Experience -> O
with -> O
Python -> B-SKILL
and -> O
Machine -> B-SKILL
Learning -> I-SKILL


In [7]:
training_rows = []

for text in df["clean_description"].head(3000):

    words, labels = create_bio_example(text)

    if "B-SKILL" in labels:
        training_rows.append({
            "tokens": words,
            "ner_tags": labels
        })

train_df = pd.DataFrame(training_rows)

print("Training examples:", len(train_df))

train_df.head()

Training examples: 2982


,tokens,ner_tags
0,"[we, are, looking, for, a, frontend, developer...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
1,"[we, are, looking, for, a, software, engineer,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,"[we, are, looking, for, a, machine, learning, ...","[O, O, O, O, O, B-SKILL, I-SKILL, O, O, O, O, ..."
3,"[we, are, looking, for, a, cloud, engineer, to...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,"[we, are, looking, for, a, frontend, developer...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."


In [8]:
print("Total training examples:", len(train_df))

skill_examples = train_df[
    train_df["ner_tags"].apply(lambda x: "B-SKILL" in x)
]

print("Examples containing skills:", len(skill_examples))

Total training examples: 2982
Examples containing skills: 2982


In [9]:
label2id = {
    "O": 0,
    "B-SKILL": 1,
    "I-SKILL": 2
}

id2label = {
    0: "O",
    1: "B-SKILL",
    2: "I-SKILL"
}

In [10]:
train_df["ner_tags"] = train_df["ner_tags"].apply(
    lambda tags: [label2id[tag] for tag in tags]
)

train_df.head()

,tokens,ner_tags
0,"[we, are, looking, for, a, frontend, developer...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,"[we, are, looking, for, a, software, engineer,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,"[we, are, looking, for, a, machine, learning, ...","[0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,"[we, are, looking, for, a, cloud, engineer, to...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,"[we, are, looking, for, a, frontend, developer...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [11]:
dataset = Dataset.from_pandas(train_df)

print(dataset)

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 2982
})


In [12]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2385
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 597
    })
})


In [13]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [40]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])

            else:
                # If the word is split into subwords,
                # convert B-SKILL to I-SKILL
                if labels[word_idx] == label2id["B-SKILL"]:
                    label_ids.append(label2id["I-SKILL"])
                else:
                    label_ids.append(labels[word_idx])

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

In [41]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

print(tokenized_dataset)

Map:   0%|          | 0/2385 [00:00<?, ? examples/s]

Map:   0%|          | 0/597 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2385
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 597
    })
})


In [42]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [43]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [44]:
training_args = TrainingArguments(
    output_dir="./skill_bert_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none"
)

In [45]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

In [46]:
trainer.train()

D:\Anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.000693,0.000216
2,0.000327,0.000103
3,0.000140,0.000083


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

D:\Anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

D:\Anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=897, training_loss=0.012919489047702905, metrics={'train_runtime': 1117.4941, 'train_samples_per_second': 6.403, 'train_steps_per_second': 0.803, 'total_flos': 109717924550400.0, 'train_loss': 0.012919489047702905, 'epoch': 3.0})

In [66]:
trainer.model.save_pretrained(
    "./skill_bert_model_final"
)

tokenizer.save_pretrained(
    "./skill_bert_model_final"
)

print("Final model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved successfully.


In [67]:
model_path = "./skill_bert_model_final"

model = AutoModelForTokenClassification.from_pretrained(
    model_path
)

tokenizer = AutoTokenizer.from_pretrained(
    model_path
)

print("Fine-tuned model loaded successfully!")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Fine-tuned model loaded successfully!


In [68]:
skill_ner = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

In [47]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_path = "./skill_bert_model/checkpoint-897"

model = AutoModelForTokenClassification.from_pretrained(model_path)

tokenizer = AutoTokenizer.from_pretrained(model_path)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Model loaded successfully!


In [51]:
save_strategy="epoch"

In [52]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_path = "./skill_bert_model/checkpoint-897"

model = AutoModelForTokenClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Model loaded successfully!


In [57]:
from transformers import pipeline

ner_model = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

print("Baseline BERT loaded successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline BERT loaded successfully!


In [61]:
def clean_skill_results(results):

    cleaned = []

    for entity in results:

        word = entity["word"].strip()

        # Remove punctuation-only entities
        if not any(char.isalnum() for char in word):
            continue

        if entity["entity_group"] == "SKILL":
            cleaned.append({
                "skill": word,
                "confidence": round(float(entity["score"]), 3)
            })

    return cleaned

In [62]:
cleaned_results = clean_skill_results(results)

for entity in cleaned_results:
    print(
        entity["skill"],
        "-> SKILL ->",
        entity["confidence"]
    )

python -> SKILL -> 1.0
sql -> SKILL -> 1.0
machine learning -> SKILL -> 0.999
aws -> SKILL -> 1.0
docker -> SKILL -> 1.0
power bi -> SKILL -> 0.999


In [71]:
text = "Experience with Python, SQL and AWS."

results = skill_ner(text)

for entity in results:

    word = entity["word"].strip()

    if not any(char.isalnum() for char in word):
        continue

    print(
        word,
        "->",
        entity["entity_group"],
        "->",
        round(float(entity["score"]), 3)
    )

python, -> SKILL -> 0.753
sql -> SKILL -> 1.0
aws -> SKILL -> 0.999


In [72]:
text = """
Experience in Machine Learning,
Natural Language Processing,
Data Science and Power BI.
"""

results = skill_ner(text)

for entity in results:

    word = entity["word"].strip()

    if not any(char.isalnum() for char in word):
        continue

    print(
        word,
        "->",
        entity["entity_group"],
        "->",
        round(float(entity["score"]), 3)
    )

machine learning -> SKILL -> 0.997
natural language processing -> SKILL -> 0.932
data science -> SKILL -> 0.978
power bi -> SKILL -> 0.996


In [73]:
text = """
We are looking for a Data Scientist with experience
in Python, SQL, Machine Learning, AWS, Docker and Power BI.
"""

results = skill_ner(text)

print("Extracted Skills:")
print("------------------")

for entity in results:

    word = entity["word"].strip()

    if not any(char.isalnum() for char in word):
        continue

    if entity["entity_group"] == "SKILL":

        print(
            word,
            "-> SKILL ->",
            round(float(entity["score"]), 3)
        )

Extracted Skills:
------------------
python -> SKILL -> 1.0
sql -> SKILL -> 1.0
machine learning -> SKILL -> 0.999
aws -> SKILL -> 1.0
docker -> SKILL -> 1.0
power bi -> SKILL -> 0.999


In [74]:
ner_model = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

print("Baseline BERT loaded successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline BERT loaded successfully!


In [75]:
text = "Experience with Python, SQL and AWS."

baseline = ner_model(text)
fine_tuned = skill_ner(text)

print("BASELINE BERT")
print("----------------")

for entity in baseline:
    print(
        entity["word"],
        "->",
        entity["entity_group"]
    )

print("\nFINE-TUNED DISTILBERT")
print("----------------------")

for entity in fine_tuned:

    word = entity["word"].strip()

    if not any(char.isalnum() for char in word):
        continue

    print(
        word,
        "->",
        entity["entity_group"]
    )

BASELINE BERT
----------------
Python -> MISC
SQL -> MISC
AWS -> MISC

FINE-TUNED DISTILBERT
----------------------
python, -> SKILL
sql -> SKILL
aws -> SKILL


In [76]:
sample_df = df.head(20).copy()

def extract_skills_transformer(text):

    results = skill_ner(str(text))

    extracted = []

    for entity in results:

        word = entity["word"].strip()

        # Remove punctuation-only predictions
        if not any(char.isalnum() for char in word):
            continue

        if entity["entity_group"] == "SKILL":

            extracted.append(word)

    # Remove duplicates while preserving order
    extracted = list(dict.fromkeys(extracted))

    return extracted

In [77]:
sample_df["transformer_skills"] = (
    sample_df["clean_description"]
    .apply(extract_skills_transformer)
)

In [96]:
sample_df[
    ["job_title", "transformer_skills"]
]

,job_title,transformer_skills
0,Frontend Developer,"[git, javascript]"
1,Software Engineer,"[git, python, sql]"
2,Machine Learning Engineer,"[machine learning, pytorch, docker, python, tensorflow, aws]"
3,Cloud Engineer,"[docker, kubernetes, azure, aws]"
4,Frontend Developer,"[git, javascript]"
5,Data Analyst,"[python, excel, sql, pandas, tableau, power bi]"
6,Machine Learning Engineer,"[machine learning, python, pytorch, sql, tensorflow]"
7,DevOps Engineer,"[aws, kubernetes, git]"
8,QA Engineer,"[sql, python]"
9,Business Analyst,"[power bi, sql, tableau, excel]"


In [97]:
pd.set_option("display.max_colwidth", None)

sample_df["skills"] = sample_df["transformer_skills"].apply(
    lambda x: ", ".join(x)
)

sample_df[["job_title", "skills"]].head(20)

,job_title,skills
0,Frontend Developer,"git, javascript"
1,Software Engineer,"git, python, sql"
2,Machine Learning Engineer,"machine learning, pytorch, docker, python, tensorflow, aws"
3,Cloud Engineer,"docker, kubernetes, azure, aws"
4,Frontend Developer,"git, javascript"
5,Data Analyst,"python, excel, sql, pandas, tableau, power bi"
6,Machine Learning Engineer,"machine learning, python, pytorch, sql, tensorflow"
7,DevOps Engineer,"aws, kubernetes, git"
8,QA Engineer,"sql, python"
9,Business Analyst,"power bi, sql, tableau, excel"


In [86]:
sample_df.to_csv(
    "final_transformer_skill_results.csv",
    index=False
)

print("Final results saved successfully.")

Final results saved successfully.


In [87]:
print("Model:", model_name)
print("Labels:", id2label)
print("Training examples:", len(train_df))
print("Training epochs:", 3)
print("Model type: Fine-tuned DistilBERT")

Model: distilbert-base-uncased
Labels: {0: 'O', 1: 'B-SKILL', 2: 'I-SKILL'}
Training examples: 2982
Training epochs: 3
Model type: Fine-tuned DistilBERT
